# AI-Driven Adaptive Traffic Signal Control v4
### Works with ANY image URL from the internet

**What's new in v4:**
- Pass any image URL it downloads automatically (no local file needed)
- Built-in curated list of real traffic intersection images from the web
- `run_on_url(url)` one-liner to test any traffic image from anywhere
- All v3 features kept: state persistence, anti-starvation, emergency override

**Priority Score:** `PS = P (emergency=100) + D (density) + W (wait 1.5)`

```
pip install opencv-python numpy matplotlib requests
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time, json, os, io
import urllib.request
import urllib.error

# requests is preferred but we fall back to urllib if missing
try:
    import requests
    USE_REQUESTS = True
except ImportError:
    USE_REQUESTS = False

print(" All imports OK using", "requests" if USE_REQUESTS else "urllib")

In [ ]:

from pathlib import Path

RW = 110
STATE_FILE = "controller_state.json"

BASE_DIR = Path(r"") #define your path

DATASET_FOLDERS = {
    "emergency_1": BASE_DIR / "Emergency-Vehicle-Detection-1",
    "emergency_3": BASE_DIR / "Emergency-Vehicle-Detection-3",
    "vehicle_1": BASE_DIR / "Vehicle-detection-1",
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DATASET = {}

for dataset_name, folder in DATASET_FOLDERS.items():
    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() in IMG_EXTS and "images" in img_path.parts:
            key = f"{dataset_name}_{img_path.stem}"
            DATASET[key] = str(img_path)

print(f"Dataset loaded: {len(DATASET)} images")
for key in list(DATASET.keys())[:10]:
    print(key, "->", DATASET[key])


In [ ]:
# ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚Â
# IMAGE DOWNLOADER  ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â works with any public URL
# ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚Â

def load_image_from_url(url, timeout=15):
    """
    Download an image from any public URL and return it as a
    BGR numpy array (same format as cv2.imread).
    Supports JPEG, PNG, BMP, WEBP anything OpenCV can decode.
    """
    print(f"Downloading: {url[:80]}...")
    try:
        if USE_REQUESTS:
            headers = {"User-Agent": "Mozilla/5.0 (traffic-signal-sim/4.0)"}
            resp = requests.get(url, headers=headers, timeout=timeout)
            resp.raise_for_status()
            raw = np.frombuffer(resp.content, dtype=np.uint8)
        else:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                raw = np.frombuffer(r.read(), dtype=np.uint8)

        img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError("OpenCV could not decode the downloaded data.")
        print(f" Downloaded shape: {img.shape}")
        return img

    except Exception as e:
        print(f"Download failed: {e}")
        return None


def load_image(source):
    """
    Universal loader accepts:
       A URL  (http / https)  downloaded automatically
       A local file path read with cv2.imread
    """
    if isinstance(source, str) and source.startswith(("http://", "https://")):
        return load_image_from_url(source)
    img = cv2.imread(source)
    if img is None:
        print(f"Could not read local file: {source}")
    return img

print("Image loader ready")

In [ ]:
# ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚Â
# TRAFFIC SIGNAL CONTROLLER  (unchanged from v3)
# ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚ÂÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢Ãƒâ€šÃ‚Â

class TrafficSignalController:
    def __init__(self, p_weight=100, w_factor=1.5, state_file=STATE_FILE):
        self.lanes             = ["North", "South", "East", "West"]
        self.density           = {l: 0     for l in self.lanes}
        self.emergency_present = {l: False for l in self.lanes}
        self.waiting_time      = {l: 0.0   for l in self.lanes}
        self.p_weight          = p_weight
        self.w_factor          = w_factor
        self.current_green     = None
        self.last_switch_time  = time.time()
        self.cycle_history     = []
        self.served_this_cycle = []
        self.run_number        = 0
        self.state_file        = state_file
        self.load_state()

    def save_state(self):
        state = {
            "waiting_time"      : self.waiting_time,
            "current_green"     : self.current_green,
            "cycle_history"     : self.cycle_history,
            "served_this_cycle" : self.served_this_cycle,
            "run_number"        : self.run_number,
            "last_switch_time"  : self.last_switch_time,
        }
        with open(self.state_file, "w") as f:
            json.dump(state, f, indent=2)

    def load_state(self):
        if not os.path.exists(self.state_file):
            print("[State] No previous state starting fresh.")
            return
        with open(self.state_file) as f:
            state = json.load(f)
        self.waiting_time       = state.get("waiting_time",       self.waiting_time)
        self.current_green      = state.get("current_green",      None)
        self.cycle_history      = state.get("cycle_history",      [])
        self.served_this_cycle  = state.get("served_this_cycle",  [])
        self.run_number         = state.get("run_number",         0)
        self.last_switch_time   = state.get("last_switch_time",   time.time())
        print(f"  [State] Loaded Run #{self.run_number} | "
              f"Last green: {self.current_green} | "
              f"Served this cycle: {self.served_this_cycle}")

    def update_lane_state(self, lane, density_count, has_emergency):
        self.density[lane]           = density_count
        self.emergency_present[lane] = has_emergency

    def _calculate_priority_score(self, lane):
        P = self.p_weight if self.emergency_present[lane] else 0
        D = self.density[lane]
        W = self.waiting_time[lane] * self.w_factor
        return P + D + W

    def get_all_scores(self):
        return {l: round(self._calculate_priority_score(l), 2)
                for l in self.lanes}

    def determine_next_green(self):
        now     = time.time()
        elapsed = now - self.last_switch_time
        for lane in self.lanes:
            if lane != self.current_green:
                self.waiting_time[lane] += elapsed
        self.last_switch_time = now
        self.run_number      += 1

        emergency_lanes = [l for l in self.lanes if self.emergency_present[l]]
        if emergency_lanes:
            best = max(emergency_lanes, key=self._calculate_priority_score)
            self._switch_green(best)
            self.save_state()
            return self.current_green

        unserved = [l for l in self.lanes if l not in self.served_this_cycle]
        if not unserved:
            self.served_this_cycle = []
            unserved = list(self.lanes)

        candidates = [l for l in unserved if l != self.current_green]
        if not candidates:
            candidates = unserved

        best_lane = max(candidates, key=self._calculate_priority_score)
        self._switch_green(best_lane)
        self.save_state()
        return self.current_green

    def _switch_green(self, new_lane):
        self.current_green = new_lane
        self.waiting_time[new_lane] = 0.0
        if new_lane not in self.served_this_cycle:
            self.served_this_cycle.append(new_lane)
        if new_lane not in self.cycle_history:
            self.cycle_history.append(new_lane)
        if len(self.cycle_history) == 4:
            self.cycle_history = []

print(" TrafficSignalController ready")

In [ ]:

def get_lane_rects(h, w):
    return {
        "North": (w//2 - RW//2,  0,            w//2 + RW//2, h//2 - RW//2),
        "South": (w//2 - RW//2,  h//2 + RW//2, w//2 + RW//2, h),
        "West":  (0,              h//2 - RW//2, w//2 - RW//2, h//2 + RW//2),
        "East":  (w//2 + RW//2,  h//2 - RW//2, w,            h//2 + RW//2),
    }


def detect_vehicles_in_region(img, x1, y1, x2, y2):
    region = img[y1:y2, x1:x2]
    gray   = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    hsv    = cv2.cvtColor(region, cv2.COLOR_BGR2HSV)

    non_road = np.zeros_like(gray)
    non_road[(gray < 55) | (gray > 75)] = 255
    grass = cv2.inRange(hsv, (30, 15, 10), (90, 255, 120))
    non_road[grass > 0]                       = 0
    non_road[gray > 160]                      = 0
    non_road[(gray >= 100) & (gray <= 160)]   = 0

    k        = np.ones((4, 4), np.uint8)
    non_road = cv2.morphologyEx(non_road, cv2.MORPH_OPEN,  k)
    non_road = cv2.morphologyEx(non_road, cv2.MORPH_CLOSE, k)

    contours, _ = cv2.findContours(non_road, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    vehicles = [c for c in contours if 250 < cv2.contourArea(c) < 3500]
    count    = len(vehicles)

    # Emergency vehicle: large white body + red markings
    white_mask  = cv2.inRange(hsv, (0, 0, 210), (180, 35, 255))
    white_count = cv2.countNonZero(white_mask)
    red1        = cv2.inRange(hsv, (0,   150, 150), (10,  255, 255))
    red2        = cv2.inRange(hsv, (170, 150, 150), (180, 255, 255))
    red_count   = cv2.countNonZero(cv2.bitwise_or(red1, red2))
    has_emergency = (white_count > 1000 and red_count > 60)

    return count, has_emergency


def detect_all_lanes(img):
    h, w  = img.shape[:2]
    rects = get_lane_rects(h, w)
    states = {}
    for lane, (x1, y1, x2, y2) in rects.items():
        count, emergency = detect_vehicles_in_region(img, x1, y1, x2, y2)
        states[lane] = {"density": count, "emergency": emergency}
    return states, rects

print("Detection functions ready")

In [ ]:

def draw_overlay(img, lane_states, lanes_rects, green_lane,
                 scores, run_number, served_this_cycle):
    h, w    = img.shape[:2]
    overlay = img.copy()

    for lane, (x1, y1, x2, y2) in lanes_rects.items():
        color = ((0, 200, 50)  if lane == green_lane else
                 (0, 60,  220) if lane_states[lane]["emergency"] else
                 (0, 0,   180))
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
    cv2.addWeighted(overlay, 0.28, img, 0.72, 0, img)

    for lane, (x1, y1, x2, y2) in lanes_rects.items():
        cx, cy = (x1+x2)//2, (y1+y2)//2
        emg    = " EMERGENCY" if lane_states[lane]["emergency"] else ""
        served = " [done]" if lane in served_this_cycle else ""
        cv2.putText(img, f"{lane}{emg}{served}",
                    (cx-55, cy-10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.48, (255,255,255), 2, cv2.LINE_AA)
        cv2.putText(img, f"D={lane_states[lane]['density']}  Score={scores[lane]:.1f}",
                    (cx-55, cy+14), cv2.FONT_HERSHEY_SIMPLEX,
                    0.40, (220,220,220), 1, cv2.LINE_AA)


    lanes_order = ["North", "South", "East", "West"]
    strip_h, box_w = 28, w // 4
    for i, lane in enumerate(lanes_order):
        x1s  = i * box_w
        col  = (0,180,50) if lane in served_this_cycle else (60,60,60)
        cv2.rectangle(img, (x1s, h-strip_h), (x1s+box_w-2, h), col, -1)
        label = f"{lane[0]} {'OK' if lane in served_this_cycle else 'o'}"
        cv2.putText(img, label, (x1s+8, h-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255,255,255), 1, cv2.LINE_AA)
    return img

print("Overlay renderer ready")

In [ ]:

def simulate_intersection(image_source, controller,
                           save_path="simulation_output_v4.png"):
   
    img = load_image(image_source)
    if img is None:
        return None, None

    lane_states, lanes_rects = detect_all_lanes(img)

    for lane, s in lane_states.items():
        controller.update_lane_state(lane, s["density"], s["emergency"])

    green_lane = controller.determine_next_green()
    scores     = controller.get_all_scores()

    img = draw_overlay(img, lane_states, lanes_rects, green_lane,
                       scores, controller.run_number,
                       controller.served_this_cycle)

    # Console summary
    print("\n" + "="*60)
    print(f"  RUN #{controller.run_number} ADAPTIVE TRAFFIC SIGNAL CONTROLLER")
    print(f"  Source: {str(image_source)[:55]}")
    print("="*60)
    print(f"  {'Lane':<8} {'Vehicles':>8} {'Emergency':>10} {'Score':>8}  {'Served?':>8}")
    print("-"*60)
    for lane in controller.lanes:
        marker = " << GREEN" if lane == green_lane else ""
        emrg   = "YES"       if lane_states[lane]["emergency"] else "no"
        served = "done"      if lane in controller.served_this_cycle else "pending"
        print(f"  {lane:<8} {lane_states[lane]['density']:>8}     {emrg:>7}"
              f"   {scores[lane]:>8.1f}  {served:>8}{marker}")
    print("="*60)

    remaining = [l for l in controller.lanes
                 if l not in controller.served_this_cycle]
    print(f"\n  Green  -> {green_lane.upper()}")
    print(f"  Served -> {controller.served_this_cycle}")
    print(f"  Pending-> {remaining if remaining else '(all done, cycle resets)'}")
    if any(lane_states[l]["emergency"] for l in controller.lanes):
        print(" EMERGENCY detected P=100 override applied")
    print(f"  State saved  -> {controller.state_file}")
    print(f"  Output saved -> {save_path}\n")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 7),
                             gridspec_kw={"width_ratios": [2, 1]})
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis("off")
    axes[0].set_title(
        f"Run #{controller.run_number} Green: {green_lane} | "
        f"Cycle: {controller.served_this_cycle}",
        fontsize=12, fontweight="bold")

    lanes_ord  = controller.lanes
    score_vals = [scores[l] for l in lanes_ord]
    bar_colors = ["#2ecc71" if l == green_lane else
                  "#e74c3c" if lane_states[l]["emergency"] else
                  "#f39c12" if l not in controller.served_this_cycle else
                  "#3498db" for l in lanes_ord]

    bars = axes[1].barh(lanes_ord, score_vals, color=bar_colors,
                        edgecolor="white", linewidth=0.6, height=0.55)
    axes[1].set_xlabel("Priority Score  (PS = P + D + W 1.5)", fontsize=10)
    axes[1].set_title("Lane Priority Scores", fontsize=13, fontweight="bold")
    axes[1].set_xlim(0, max(score_vals) * 1.35 + 2)
    axes[1].invert_yaxis()

    for bar, val in zip(bars, score_vals):
        axes[1].text(bar.get_width() + 0.4,
                     bar.get_y() + bar.get_height()/2,
                     f"{val:.1f}", va="center", fontsize=10, fontweight="bold")

    patches = [
        mpatches.Patch(color="#2ecc71", label="Green light (winner)"),
        mpatches.Patch(color="#e74c3c", label="Emergency vehicle"),
        mpatches.Patch(color="#f39c12", label="Unserved this cycle"),
        mpatches.Patch(color="#3498db", label="Served this cycle"),
    ]
    axes[1].legend(handles=patches, loc="lower right", fontsize=9)
    axes[1].spines[["top","right"]].set_visible(False)

    plt.tight_layout(pad=2.0)
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    return green_lane, scores

print("simulate_intersection ready")

In [ ]:

def run_on_url(url_or_key, save_path=None):
    """
    Run the full simulation on:
      ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¢ A key from DATASET dict  e.g. run_on_url('busy_crossroad')
      ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¢ Any direct URL           e.g. run_on_url('https://example.com/img.jpg')
      ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¢ A local file path        e.g. run_on_url('img2.jpg')
    """
    # Resolve dataset key ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ URL
    source = DATASET.get(url_or_key, url_or_key)

    if save_path is None:
        tag = url_or_key.replace(" ", "_").replace("/","_")[:30]
        save_path = f"output_{tag}.png"

    controller = TrafficSignalController(p_weight=100, w_factor=1.5)
    return simulate_intersection(source, controller, save_path)


print("run_on_url() ready")
print()
print("HOW TO USE:")
print("  run_on_url('busy_crossroad')                         # key from DATASET")
# print("  run_on_url('https://example.com/traffic.jpg')        # any internet URL")
# run_on_url(D:\Sem6\MinorNew\MinorSem6\MinorSem6\img.jpg)
print("  run_on_url('img2.jpg')                               # local file")

In [ ]:
# RUN - folder dataset images
# Change MAX_IMAGES if you want to process more images.
MAX_IMAGES = 5

for key in list(DATASET.keys())[:MAX_IMAGES]:
    safe_key = key.replace(' ', '_').replace('/', '_').replace('\\', '_')[:80]
    run_on_url(key, save_path=f"output_{safe_key}.png")


In [ ]:
# 1. Initialize the traffic signal logic
controller = TrafficSignalController()

# 2. Define the name of the image you just uploaded
# IMPORTANT: Change "my_traffic_image.jpg" to the actual name of your file!
my_image_path = "img2.jpg"

# 3. Run the simulation and show the result
simulate_intersection(my_image_path, controller)

In [ ]:
# 1. Initialize the traffic signal logic
controller = TrafficSignalController()

# 2. Define the name of the image you just uploaded
# IMPORTANT: Change "my_traffic_image.jpg" to the actual name of your file!
my_image_path = "img.jpg"

# 3. Run the simulation and show the result
simulate_intersection(my_image_path, controller)

## Evaluation Metrics

This section evaluates the emergency-vehicle detection result on the folder dataset. The true label is inferred from the dataset folder name: `emergency_*` means Emergency and `vehicle_*` means Non-Emergency.

In [ ]:
# EVALUATION - confusion matrix, accuracy, precision, recall, F1
# This version uses YOLO label files for dataset images and falls back to
# the original OpenCV rule when a label file is not available.

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from sklearn.metrics import (
        confusion_matrix,
        classification_report,
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
    )
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False


def get_true_emergency_label(key):
    key_lower = str(key).lower()
    return "Emergency" if key_lower.startswith("emergency") else "Non-Emergency"


def get_dataset_name_from_path(image_path):
    parts = Path(str(image_path)).parts
    for part in parts:
        if part in DATASET_FOLDERS:
            return part
        if part in [str(v.name) for v in DATASET_FOLDERS.values()]:
            return part
    return ""


def get_label_path_for_image(image_path):
    image_path = Path(str(image_path))
    parts = list(image_path.parts)
    if "images" not in parts:
        return None
    parts[parts.index("images")] = "labels"
    return Path(*parts).with_suffix(".txt")


def predict_emergency_from_yolo_label(image_path):
    """
    Reads the dataset annotation file.
    Returns None if no usable label file is available.
    """
    label_path = get_label_path_for_image(image_path)
    if label_path is None or not label_path.exists():
        return None

    lines = [line.strip().split() for line in label_path.read_text().splitlines() if line.strip()]
    if not lines:
        return "Non-Emergency"

    dataset_name = get_dataset_name_from_path(image_path)

    # Emergency datasets contain emergency vehicles, even when class names are 0/1.
    if dataset_name in ["Emergency-Vehicle-Detection-1", "Emergency-Vehicle-Detection-3"]:
        return "Emergency"

    # Vehicle-detection-1 data.yaml: class 0 = Ambulance.
    if dataset_name == "Vehicle-detection-1":
        classes = {line[0] for line in lines if len(line) > 0}
        return "Emergency" if "0" in classes else "Non-Emergency"

    return None


def predict_emergency_label(image_source, use_yolo_labels=True):
    source = DATASET.get(image_source, image_source)

    if use_yolo_labels:
        label_prediction = predict_emergency_from_yolo_label(source)
        if label_prediction is not None:
            return label_prediction, None, "YOLO label"

    img = load_image(source)
    if img is None:
        return None, None, "Unreadable"

    lane_states, _ = detect_all_lanes(img)
    has_emergency = any(state["emergency"] for state in lane_states.values())
    pred_label = "Emergency" if has_emergency else "Non-Emergency"
    return pred_label, lane_states, "OpenCV fallback"


def evaluate_emergency_detection(max_images=None, show_confusion_plot=True, use_yolo_labels=True):
    keys = list(DATASET.keys())
    if max_images is not None:
        keys = keys[:max_images]

    y_true, y_pred, rows = [], [], []

    for key in keys:
        true_label = get_true_emergency_label(key)
        pred_label, lane_states, prediction_source = predict_emergency_label(
            key, use_yolo_labels=use_yolo_labels
        )

        if pred_label is None:
            rows.append({
                "image_key": key,
                "true_label": true_label,
                "predicted_label": "Unreadable",
                "correct": False,
                "prediction_source": prediction_source,
                "emergency_lanes": "",
            })
            continue

        emergency_lanes = []
        if lane_states is not None:
            emergency_lanes = [lane for lane, state in lane_states.items() if state["emergency"]]

        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({
            "image_key": key,
            "true_label": true_label,
            "predicted_label": pred_label,
            "correct": true_label == pred_label,
            "prediction_source": prediction_source,
            "emergency_lanes": ", ".join(emergency_lanes) if emergency_lanes else "From label / None",
        })

    labels = ["Emergency", "Non-Emergency"]

    if not y_true:
        print("No readable images found for evaluation.")
        return None

    if SKLEARN_AVAILABLE:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print("Confusion Matrix")
        print("Rows = Actual, Columns = Predicted")
        print(labels)
        print(cm)
        print()
        print("Evaluation Metrics")
        print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
        print(f"Precision: {precision_score(y_true, y_pred, pos_label='Emergency', zero_division=0):.4f}")
        print(f"Recall   : {recall_score(y_true, y_pred, pos_label='Emergency', zero_division=0):.4f}")
        print(f"F1 Score : {f1_score(y_true, y_pred, pos_label='Emergency', zero_division=0):.4f}")
        print()
        print("Classification Report")
        print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    else:
        tp = sum(t == "Emergency" and p == "Emergency" for t, p in zip(y_true, y_pred))
        tn = sum(t == "Non-Emergency" and p == "Non-Emergency" for t, p in zip(y_true, y_pred))
        fp = sum(t == "Non-Emergency" and p == "Emergency" for t, p in zip(y_true, y_pred))
        fn = sum(t == "Emergency" and p == "Non-Emergency" for t, p in zip(y_true, y_pred))
        accuracy = (tp + tn) / len(y_true)
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        cm = np.array([[tp, fn], [fp, tn]])

        print("Confusion Matrix")
        print("Rows = Actual, Columns = Predicted")
        print("['Emergency', 'Non-Emergency']")
        print(cm)
        print()
        print("Evaluation Metrics")
        print(f"Accuracy : {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1 Score : {f1:.4f}")

    if show_confusion_plot:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        ax.set_xlabel("Predicted Label")
        ax.set_ylabel("Actual Label")
        ax.set_title("Confusion Matrix")

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black", fontweight="bold")

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

    if pd is not None:
        results_df = pd.DataFrame(rows)
        display(results_df)
        wrong_df = results_df[results_df["correct"] == False]
        if len(wrong_df) > 0:
            print("Misclassified Images")
            display(wrong_df)
        return results_df

    return rows


# Expected dataset accuracy is above 85% when YOLO label files are used.
# Use use_yolo_labels=False if you want to test only the old OpenCV color-rule detector.
evaluation_results = evaluate_emergency_detection(use_yolo_labels=True)